In [14]:
print("Installing necessary libraries...")
!pip install -q langchain langchain-openai openai chromadb gradio python-dotenv tiktoken langchain-community
print("Libraries installed successfully!")

Installing necessary libraries...
Libraries installed successfully!



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# !pip install --upgrade openai
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(api_key=openai_api_key)
print("OpenAI client successfully configured.")

print(openai_api_key[:15])

OpenAI client successfully configured.


In [16]:
# Let's import Langchain components
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader
from langchain.chains import RetrievalQAWithSourcesChain 

In [17]:
DATA_FILE_PATH = r"C:\Users\saria\Desktop\LLM+and+Agentic+AI+Bootcamp+Materials\LLM and Agentic AI Bootcamp Materials\Module 7 - RAG & Langchain\eleven_madison_park_data.txt"
print(f"Data file path set to: {DATA_FILE_PATH}")

Data file path set to: C:\Users\saria\Desktop\LLM+and+Agentic+AI+Bootcamp+Materials\LLM and Agentic AI Bootcamp Materials\Module 7 - RAG & Langchain\eleven_madison_park_data.txt


In [18]:
print(f"Attempting to load data from: {DATA_FILE_PATH}")

loader = TextLoader(DATA_FILE_PATH, encoding = "utf-8")

raw_documents = loader.load()
print(f"Successfully loaded {len(raw_documents)} document(s).")


Attempting to load data from: C:\Users\saria\Desktop\LLM+and+Agentic+AI+Bootcamp+Materials\LLM and Agentic AI Bootcamp Materials\Module 7 - RAG & Langchain\eleven_madison_park_data.txt
Successfully loaded 1 document(s).


In [19]:
print(raw_documents[0].page_content[:500] + "...")

Source: https://www.elevenmadisonpark.com/
Title: Eleven Madison Park
Content:
Book on Resy
---END OF SOURCE---

Source: https://www.elevenmadisonpark.com/careers
Title: Careers — Eleven Madison Park
Content:
Join Our Team Eleven Madison Park ▾ All Businesses Eleven Madison Park Clemente Bar Daniel Humm Hospitality Filter Categories Culinary Pastry Wine & Beverage Dining Room Office & Admin Other Job Types Full Time Part Time Compensation Salary Hourly Apply filters OPEN OPPORTUNITIES Staff Acco...


### SPLITTING DOCUMENTS (CHUNKING) WITH LANGCHAIN TEXT SPLITTER

In [20]:
print("\nSplitting the loaded document into smaller chunks...")

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,
                                               chunk_overlap = 150)

documents = text_splitter.split_documents(raw_documents)

if not documents:
    raise ValueError("Error: Splitting resulted in zero documents. Check the input file and splitter settings.")
print(f"Document split into {len(documents)} chunks.")


Splitting the loaded document into smaller chunks...
Document split into 38 chunks.


In [21]:
documents

[Document(metadata={'source': 'C:\\Users\\saria\\Desktop\\LLM+and+Agentic+AI+Bootcamp+Materials\\LLM and Agentic AI Bootcamp Materials\\Module 7 - RAG & Langchain\\eleven_madison_park_data.txt'}, page_content='Source: https://www.elevenmadisonpark.com/\nTitle: Eleven Madison Park\nContent:\nBook on Resy\n---END OF SOURCE---'),
 Document(metadata={'source': 'C:\\Users\\saria\\Desktop\\LLM+and+Agentic+AI+Bootcamp+Materials\\LLM and Agentic AI Bootcamp Materials\\Module 7 - RAG & Langchain\\eleven_madison_park_data.txt'}, page_content='Source: https://www.elevenmadisonpark.com/careers\nTitle: Careers — Eleven Madison Park\nContent:'),
 Document(metadata={'source': 'C:\\Users\\saria\\Desktop\\LLM+and+Agentic+AI+Bootcamp+Materials\\LLM and Agentic AI Bootcamp Materials\\Module 7 - RAG & Langchain\\eleven_madison_park_data.txt'}, page_content="Join Our Team Eleven Madison Park ▾ All Businesses Eleven Madison Park Clemente Bar Daniel Humm Hospitality Filter Categories Culinary Pastry Wine & B

In [22]:
print("\n--- Example Chunk (Chunk 2) ---")
print(documents[2].page_content)
print("\n--- Metadata for Chunk 2 ---")
print(documents[2].metadata) # Should show {'source': 'eleven_madison_park_data.txt'}


--- Example Chunk (Chunk 2) ---
Join Our Team Eleven Madison Park ▾ All Businesses Eleven Madison Park Clemente Bar Daniel Humm Hospitality Filter Categories Culinary Pastry Wine & Beverage Dining Room Office & Admin Other Job Types Full Time Part Time Compensation Salary Hourly Apply filters OPEN OPPORTUNITIES Staff Accountant - Part Time Eleven Madison Park Part Time • Hourly ($20 - $25) Host/Reservationist Eleven Madison Park Full Time • Hourly ($24) Sous Chef Eleven Madison Park Full Time • Salary ($72K - $75K) Pastry Cook Eleven Madison Park Full Time • Hourly ($18 - $20) Kitchen Server Eleven Madison Park Full Time • Hourly ($16) plus tips Dining Room Manager Eleven Madison Park Full Time • Salary ($72K - $75K) Porter Manager Eleven Madison Park Full Time • Salary ($70K - $75K) Senior Sous Chef Eleven Madison Park Full Time • Salary ($85K - $95K) Maitre D Eleven Madison Park Full Time • Hourly ($16) plus tips Even if you don't see the opportunity you're looking for, we would sti

### EMBEDDINGS AND VECTOR STORE CREATION

In [23]:
print("Initializing OpenAI Embeddings model...")

embeddings = OpenAIEmbeddings(openai_api_key = openai_api_key)

print("OpenAI Embeddings model initialized.")
print("\nCreating ChromaDB vector store and embedding documents...")

vector_store = Chroma.from_documents(documents = documents, embedding = embeddings)

vector_count = vector_store._collection.count()
print(f"ChromaDB vector store created with {vector_count} items.")

if vector_count == 0:
    raise ValueError("Vector store creation resulted in 0 items. Check previous steps.")

Initializing OpenAI Embeddings model...
OpenAI Embeddings model initialized.

Creating ChromaDB vector store and embedding documents...


ChromaDB vector store created with 76 items.


In [24]:
stored_data = vector_store._collection.get(include=["embeddings", "documents"], limit=1)

print("First chunk text:\n", stored_data['documents'][0])
print("\nEmbedding vector:\n", stored_data['embeddings'][0])
print(f"\nFull embedding has {len(stored_data['embeddings'][0])} dimensions.")

First chunk text:
 Source: https://www.elevenmadisonpark.com/
Title: Eleven Madison Park
Content:
Book on Resy
---END OF SOURCE---

Embedding vector:
 [ 0.02330522 -0.01571015 -0.00706136 ... -0.02464633 -0.01022939
 -0.06158162]

Full embedding has 1536 dimensions.


### TESTING THE RETRIEVAL

In [25]:
print("\n--- Testing Similarity Search in Vector Store ---")
test_query = "Is there a dress code?"
print(f"Searching for documents similar to: '{test_query}'")

try:
    similar_docs = vector_store.similarity_search(test_query, k = 2)
    print(f"\nFound {len(similar_docs)} similar documents:")

    for i, doc in enumerate(similar_docs):
        print(f"\n--- Document {i+1} ---")
        # Displaying the first 700 chars for brevity
        content_snippet = doc.page_content[:700].strip() + "..."
        source = doc.metadata.get("source", "Unknown Source")
        print(f"Content Snippet: {content_snippet}")
        print(f"Source: {source}")

except Exception as e:
    print(f"An error occurred during similarity search: {e}")



--- Testing Similarity Search in Vector Store ---
Searching for documents similar to: 'Is there a dress code?'

Found 2 similar documents:

--- Document 1 ---
Content Snippet: starting at $125 per guest, as well as a full wine list that our wine team can help you select from and may also be viewed on our website. You are also welcome to bring your own special bottle of wine for your meal for a $75 per 750ml bottle corkage (4 bottle maximum) fee. We do not have a dress code. Many of our guests dress up for the occasion but wear whatever will make you most comfortable. You can reach our reservations team at any time at info@elevenmadisonpark.com . We can accommodate up to seven guests at a table in the main dining room. If you are interested in booking a table for a group larger than this, please contact our Private Dining and Special Events team at events@elevenmad...
Source: C:\Users\saria\Desktop\LLM+and+Agentic+AI+Bootcamp+Materials\LLM and Agentic AI Bootcamp Materials\Module 7 - R

### BUILDING & TESTING THE RAG CHAIN USING LANGCHAIN

In [26]:
from langchain_openai import ChatOpenAI

retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("Retriever configured successfully from vector store.")

llm = ChatOpenAI(temperature = 0, openai_api_key = openai_api_key)
print("OpenAI LLM successfully initialized.")

qa_chain = RetrievalQAWithSourcesChain.from_chain_type(llm = llm,
                                                       chain_type = "stuff",
                                                       retriever = retriever,
                                                       return_source_documents = True,  # Request the actual Document objects used
                                                       verbose = True)  # Set to True to see Langchain's internal steps (can be noisy)

print("RetrievalQAWithSourcesChain created")

Retriever configured successfully from vector store.
OpenAI LLM successfully initialized.
RetrievalQAWithSourcesChain created


In [27]:
# --- Test the Full Chain ---
print("\n--- Testing the Full RAG Chain ---")
chain_test_query = "What kind of food does Eleven Madison Park serve?"
print(f"Query: {chain_test_query}")

try:
    result = qa_chain.invoke({"question": chain_test_query})
    
    print("\n--- Answer ---")
    print(result.get("answer", "No answer generated."))

    print("\n--- Sources ---")
    print(result.get("sources", "No sources identified."))

    if "source_documents" in result:
        print("\n--- Source Document Snippets ---")
        for i, doc in enumerate(result["source_documents"]):
            content_snippet = doc.page_content[:250].strip()
            print(f"Doc {i+1}: {content_snippet}")

except Exception as e:
    print(f"\nAn error occurred while running the chain: {e}")


--- Testing the Full RAG Chain ---
Query: What kind of food does Eleven Madison Park serve?


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.

--- Answer ---
Eleven Madison Park serves a fully plant-based menu, using no animal products.


--- Sources ---
C:\Users\saria\Desktop\LLM+and+Agentic+AI+Bootcamp+Materials\LLM and Agentic AI Bootcamp Materials\Module 7 - RAG & Langchain\eleven_madison_park_data.txt, https://www.elevenmadisonpark.com/ourrestaurant

--- Source Document Snippets ---
Doc 1: Welcome to Eleven Madison Park Eleven Madison Park is a fine dining restaurant in the heart of New York City. Overlooking Madison Square Park–one of Manhattan’s most beautiful green spaces–we sit at the base of a historic Art Deco building on the cor
Doc 2: Welcome to Eleven Madison Park Eleven Madison Park is a fine dining restaurant in the heart of New York City. Overlooking Madison Square Park–one of Manhattan’s most beautiful green spaces–we sit at the base of a histo

### CREATING A GRADIO INTERFACE FOR OUR RAG CHAIN

In [28]:
!pip install --upgrade pip

  Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)


ERROR: To modify pip, please run the following command:
C:\Users\saria\Desktop\LLM+and+Agentic+AI+Bootcamp+Materials\LLM and Agentic AI Bootcamp Materials\rag_env\Scripts\python.exe -m pip install --upgrade pip

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import gradio as gr

In [ ]:
def ask_elevenmadison_assistant(user_query):

    print(f"\nProcessing Gradio query: '{user_query}'")
    if not user_query or user_query.strip() == "":
        print("--> Empty query received.")
        return "Please enter a question.", ""  
    
    try:
        result = qa_chain.invoke({"question": user_query})

        answer = result.get("answer", "Sorry, I couldn't find an answer in the provided documents.")
        sources = result.get("sources", "No specific sources identified.")

        if sources == DATA_FILE_PATH:
            sources = f"Retrieved from: {DATA_FILE_PATH}"
        elif isinstance(sources, list):  # Handle potential list of sources
            sources = ", ".join(list(set(sources)))  # Unique, comma-separated

        print(f"--> Answer generated: {answer[:100].strip()}...")
        print(f"--> Sources identified: {sources}")

        return answer.strip(), sources

    except Exception as e:
        error_message = f"An error occurred: {e}"
        print(f"--> Error during chain execution: {error_message}")
        # Return error message to the user interface
        return error_message, "Error occurred"


print("\nSetting up Gradio interface...")

with gr.Blocks(theme=gr.themes.Soft(), title="Eleven Madison Park Q&A Assistant") as demo:
    gr.Markdown(
        """
        # Eleven Madison Park - AI Q&A Assistant 💬
        Ask questions about the restaurant based on its website data.
        The AI provides answers and cites the source document.
        *(Examples: What are the menu prices? Who is the chef? Is it plant-based?)*
        """
    )

    question_input = gr.Textbox(
        label = "Your Question:",
        placeholder = "e.g., What are the opening hours on Saturday?",
        lines = 2,
    )

    with gr.Row():
        answer_output = gr.Textbox(label="Answer:", interactive=False, lines=6)  

        sources_output = gr.Textbox(label="Sources:", interactive=False, lines=2)

    with gr.Row():
        submit_button = gr.Button("Ask Question", variant="primary")
        clear_button = gr.ClearButton(components=[question_input, answer_output, sources_output], value="Clear All")

    gr.Examples(
        examples=[
            "What are the different menu options and prices?",
            "Who is the head chef?",
            "What is Magic Farms?"],
        inputs=question_input,  
        cache_examples=False, 
    )

    submit_button.click(fn = ask_elevenmadison_assistant, inputs = question_input, outputs = [answer_output, sources_output])

print("Gradio interface defined.")

print("\nLaunching Gradio app... (Stop the kernel or press Ctrl+C in terminal to quit)")
demo.launch()  


Setting up Gradio interface...


C:\Users\saria\AppData\Local\Temp\ipykernel_35580\1499619029.py:33: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Eleven Madison Park Q&A Assistant") as demo:


Gradio interface defined.

Launching Gradio app... (Stop the kernel or press Ctrl+C in terminal to quit)
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.



Processing Gradio query: 'Who is the head chef?'


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.
--> Answer generated: The head chef is Daniel Humm....
--> Sources identified: Retrieved from: C:\Users\saria\Desktop\LLM+and+Agentic+AI+Bootcamp+Materials\LLM and Agentic AI Bootcamp Materials\Module 7 - RAG & Langchain\eleven_madison_park_data.txt

Processing Gradio query: 'how many spoues 
'


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.
--> Answer generated: I don't know....
--> Sources identified: 

Processing Gradio query: 'how many recepes '


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.
--> Answer generated: I don't know the answer....
--> Sources identified: 

Processing Gradio query: 'how many recipes '


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.
--> Answer generated: I don't know....
--> Sources identified: 

Processing Gradio query: 'how many item in the menu'


> Entering 